In [1]:
from src.timeseries import get_days_and_bounds
from src.dataset_builder import create_template_nc

In [2]:
frequency = "annual"

EPOCH_DATE = "1850-01-01"
START_DATE = "2000-01-01"
END_DATE = "2101-01-01"

epoch, time, bounds = get_days_and_bounds(
    start_date=START_DATE,
    end_date=END_DATE,
    epoch_date=EPOCH_DATE,
    frequency=frequency,
)

annual_ds = create_template_nc(
    time=time,
    bounds=bounds,
    frequency=frequency,
    epoch=epoch,
)

annual_ds

<xarray.Dataset> Size: 2kB
Dimensions:             (annual_time: 101, nbounds: 2)
Coordinates:
  * annual_time         (annual_time) int64 808B 54786 55152 ... 90946 91311
  * nbounds             (nbounds) int64 16B 0 1
Data variables:
    annual_time_bounds  (annual_time, nbounds) int32 808B 54421 54786 ... 91311

In [3]:
annual_ds.to_netcdf("template_new.nc", engine="netcdf4", format="NETCDF4")


In [4]:
import subprocess

result = subprocess.run(
    ["cfchecks", "data_template_new.nc"],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)

CHECKING NetCDF FILE: data_template_new.nc
Using CF Checker Version 4.1.0
Checking against CF Version CF-1.8
Using Standard Name Table Version 93 (2026-03-17T10:53:20Z)
Using Area Type Table Version 13 (20 March 2025)
Using Standardized Region Name Table Version 5 (12 November 2024)

WARN: (2.6.1): No 'Conventions' attribute present

------------------
Checking variable: annual_time_bounds
------------------

------------------
Checking variable: annual_time
------------------

------------------
Checking variable: nbounds
------------------
WARN: (3): No standard_name or long_name attribute specified
WARN: (3.1): units attribute should be present

------------------
Checking variable: mass
------------------

ERRORS detected: 0
WARNINGS given: 3
INFORMATION messages: 0




In [ ]:
import yaml
from src.dataset_builder import add_variable

with open("variables.yml", "r") as f:
    VARS = yaml.safe_load(f)["variables"]

ds = annual_ds  # or monthly_ds

for var_key, meta in VARS.items():

    ds = add_variable(
        ds,
        var_key,
        meta=meta,
        time_dim="annual_time",
    )

ds.to_netcdf("data_template_new.nc")